In [26]:
include("../src/EBC.jl")
include("../../BeamToyModel/src/BeamToyModel.jl")

Main.BeamToyModel

In [17]:
nside_in = 32
res = Resolution(nside_in)
lmax_in = 3nside_in-1
fwhm = deg2rad(1.0)
blm_harmonic = BeamToyModel.gaussian_harmonic(lmax_in, fwhm, mmax = lmax_in)
;

In [18]:
cs = ConvolutionSky()
fieldnames(typeof(cs))
cb = ConvolutionBeam()
fieldnames(typeof(cb))
cc = ConvolutionCalculate(nside_output = nside_in, lstart = 0, mmax_calculate = 2)
fieldnames(typeof(cc))

(:nside_output, :lstart, :lstop, :mmax_calculate, :HWP)

In [19]:
nalm = Healpix.numberOfAlms(lmax_in, lmax_in)
almT = rand(ComplexF64, nalm)
almE = rand(ComplexF64, nalm)
almB = rand(ComplexF64, nalm)
cs.lmax = lmax_in
cs.alm = hcat(almT, almE, almB)
;

In [20]:
cb.lmax = lmax_in
#cb.mmax = 2
cb.blm = hcat(blm_harmonic.blmT.alm,blm_harmonic.blmE.alm,blm_harmonic.blmB.alm)
;

In [21]:
n_beam = sum(2*min(l, cb.mmax) + 1 for l in cc.lstart:cc.lstop)
D_beam = spzeros(n_beam, n_beam)
n_sky = sum(2*l + 1 for l in cc.lstart:cc.lstop)
D_sky = spzeros(n_sky, n_beam)
;

In [22]:
nptg = 10
θ = pi/3
φ = pi/4
dθ = rand(nptg)*1e-2
dφ = rand(nptg)*1e-2
ψ = rand(nptg)*2pi

10-element Vector{Float64}:
 5.955859914069821
 4.674068709102341
 0.6311490333518842
 5.904857787486881
 3.9142248943940317
 2.030192169601584
 3.600408048755378
 1.605988721800255
 2.051585420055067
 5.654190871836119

In [23]:
alphas = zeros(size(dθ))
betas = zeros(size(dθ))
gammas = zeros(size(dθ))

for i in eachindex(dθ)
    err, (alphas[i], betas[i], gammas[i]) = check_split(φ, θ, dφ[i], dθ[i], ψ[i])
    if err >= 1e0
        @show err
    end
end

In [27]:
A = local_effective_wignerD(cb, cc, alphas, betas, gammas)
B = global_wignerD(cc, φ, θ, 0)
C = B*A

9216×474 SparseMatrixCSC{ComplexF64, Int64} with 46070 stored entries:
⎡⢧⠀⠀⎤
⎢⢸⠀⠀⎥
⎢⢸⠀⠀⎥
⎢⢸⡄⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⢿⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⡀⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎣⠀⠀⡇⎦

In [28]:
C

9216×474 SparseMatrixCSC{ComplexF64, Int64} with 46070 stored entries:
⎡⢧⠀⠀⎤
⎢⢸⠀⠀⎥
⎢⢸⠀⠀⎥
⎢⢸⡄⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⢿⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⡀⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎣⠀⠀⡇⎦

In [29]:
D = local_effective_wignerD(cb, cc, φ.+dφ, θ .+dθ, ψ)

9216×474 SparseMatrixCSC{ComplexF64, Int64} with 46070 stored entries:
⎡⢧⠀⠀⎤
⎢⢸⠀⠀⎥
⎢⢸⠀⠀⎥
⎢⢸⡄⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⢿⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⡀⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎣⠀⠀⡇⎦

In [30]:
maximum(abs.(C-D))

1.0122617866210908e-13

In [13]:
cc.lstop

95

In [14]:
cc.lstop

95